# Test Open Model - Quick Sanity Check

Run all 4 prompt conditions on the first 20 questions to verify Gemma 3 4B IT generates real answers.

In [ ]:
import sys
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login

sys.path.append('../../src')

from load_dataset import load_truthfulqa
from prompts import load_prompt_template, format_prompt
from open_model import load_model, generate_answer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
print('Setup complete.')

In [ ]:
MODEL_NAME = 'google/gemma-3-4b-it'
N = 3

df = load_truthfulqa('../../data/TruthfulQA.csv').head(N)
print(f'Loaded {len(df)} questions')

model, tokenizer, device = load_model(MODEL_NAME, token=hf_token)
print(f'Model: {MODEL_NAME}')
print(f'Device: {device}')

In [ ]:
prompt_configs = {
    'baseline': '../../prompts/baseline.txt',
    'adversarial_force_wrong': '../../prompts/adversarial_force_wrong.txt',
    'adversarial_popular_myth': '../../prompts/adversarial_popular_myth.txt',
    'adversarial_conspiracy': '../../prompts/adversarial_conspiracy.txt',
}

for prompt_name, prompt_path in prompt_configs.items():
    template = load_prompt_template(prompt_path)
    print(f"\n{'='*60}")
    print(f'  {prompt_name}')
    print(f"{'='*60}")

    empty_count = 0
    for i in range(N):
        question = df.iloc[i]['Question']
        prompt = format_prompt(template, question)
        answer = generate_answer(model, tokenizer, prompt, device)
        status = 'OK' if answer else 'EMPTY'
        if not answer:
            empty_count += 1
        print(f'[{i+1:2d}] [{status}] Q: {question[:60]}')
        print(f'          A: {answer[:120] if answer else "(empty)"}')
        print()

    print(f'--- {prompt_name}: {N - empty_count}/{N} OK, {empty_count}/{N} EMPTY ---')